# 🤖 Orion Knowledge Assistant
## Construyendo un agente KA con Agent Bricks

---

> **DATABRICKS · MOSAIC AI · LABORATORIO**
>
> En este laboratorio construirás un agente conversacional sobre documentación técnica usando el enfoque **declarativo** de Agent Bricks — sin escribir código de retrieval, sin configurar embeddings manualmente, sin gestionar índices.

| ⏱ ~40 minutos | 🛠 Databricks Free Trial | ⚙ Serverless Compute v5 |
|:---:|:---:|:---:|

---

## 🤖 El escenario: Orion A1

La empresa **Orion Robotics** fabrica el robot humanoide **Orion A1** para uso industrial. Sus ingenieros de campo necesitan respuestas rápidas y precisas sobre:

- Procedimientos de mantenimiento y recalibración
- Verificación de checksums de firmware
- Cumplimiento de normas de seguridad (ISO 13849-1)
- Resolución de problemas en campo

Hoy, esa información está dispersa en manuales técnicos, guías de compliance y documentación interna almacenada en Databricks. **Nuestro objetivo**: construir el **Orion Knowledge Assistant (OKA)**, un agente que responda preguntas técnicas basándose exclusivamente en esa documentación, con citas a la fuente.

---

## 🎯 Objetivos del laboratorio

Al finalizar este lab serás capaz de:

| # | Objetivo |
|---|----------|
| 1 | Crear un agente Knowledge Assistant desde la UI de Agent Bricks |
| 2 | Configurar fuentes de conocimiento desde Unity Catalog Volumes |
| 3 | Probar el agente e interpretar sus respuestas y citas |
| 4 | Identificar respuestas fuera del contexto y entender cómo el agente las maneja |
| 5 | Mejorar la calidad del agente usando labeled data y el ciclo ALHF |

---

## ✅ Requisitos

Antes de comenzar, verifica lo siguiente:

- [ ] Tienes acceso a un workspace de **Databricks Free Trial** (o workspace con Agent Bricks Preview habilitado)
- [ ] El compute está configurado como **Serverless** con **environment version 5**
- [ ] Puedes navegar a la sección **Agents** en el menú lateral izquierdo

> 💡 **Nota:** Agent Bricks está actualmente en **Preview (Beta)**. Asegúrate de que tu workspace tenga esta feature habilitada. En el Free Trial viene activa por defecto.

---

## ⚙️ Setup del workspace

> 📌 **Paso previo requerido:** Antes de ejecutar este notebook, asegúrate de haber ejecutado el notebook **`Lab_AgentBricks_Setup_Volume`** para crear el Volume y copiar los documentos de Orion. Si ya lo hiciste, continúa aquí.

La siguiente celda verifica que tu entorno está listo para el laboratorio.

In [ ]:
import os, sys

print(f"✅ Python {sys.version.split()[0]}")
print()

# Verificar que el Volume existe y tiene documentos
CATALOG = "workspace"
SCHEMA  = "default"
VOLUME  = "orion_docs"
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print("📋 Verificando entorno...")
print()

# Verificar Volume
if os.path.exists(volume_path):
    files = [f for f in os.listdir(volume_path) if not f.startswith(".")]
    if files:
        print(f"✅ Volume encontrado: {volume_path}")
        print(f"✅ Documentos disponibles: {len(files)} archivo(s)")
        for f in sorted(files):
            print(f"   📄 {f}")
    else:
        print(f"⚠️  Volume existe pero está vacío.")
        print(f"   Ejecuta primero OKA_00_Setup_Volume para copiar los documentos.")
else:
    print(f"❌ Volume no encontrado: {volume_path}")
    print(f"   Ejecuta primero OKA_00_Setup_Volume antes de continuar.")

print()
print("🚀 ¡Listo para comenzar el laboratorio!")

---
# Parte A — Crear el agente Orion KA
> ⏱ **~15 minutos** · Todo desde la UI de Databricks, sin código

En esta parte usaremos la interfaz de Agent Bricks para crear el agente de forma declarativa. Vamos a aprender qué significa que sea "declarativo": tú describes **qué** quieres, Agent Bricks decide **cómo** construirlo.

### Paso A1 — Navegar a Agents

1. En el menú lateral izquierdo de Databricks, busca **Agents**
2. Haz clic en el botón **Create Agent**
3. En la pantalla de selección, haz clic en la tarjeta **Knowledge Assistant**

![Knowledge Assistant](https://raw.githubusercontent.com/yaliagap/01-agentbricks-knowledge-assistant/main/img/ka_selection.png)

> 💡 **¿Por qué Knowledge Assistant?**  
> Es el brick optimizado para RAG sobre documentación. Maneja automáticamente el parsing, chunking, embeddings y generación de citas. Ideal para el caso de Orion donde la fuente de verdad son documentos técnicos.

---

### Paso A2 — Configurar el agente

Completa el formulario con los siguientes datos. Puedes copiar los textos directamente desde aquí:

#### Nombre
```
Orion_KA_Agent
```

#### Descripción del agente
```
Orion KA Agent (OKA) helps engineers and technicians quickly find
accurate answers from Orion's internal manuals, maintenance guides, and safety
documents. It delivers clear, verified responses with source references,
reducing search time and ensuring consistent, reliable information across teams.
```

#### Fuente de conocimiento
- **Knowledge source type**: `Files in a Volume`
- **Source**: Selecciona el Unity Catalog Volume que contiene los documentos de Orion
- **Knowledge source name**: `Company Documents`
- **Content description**:
```
Contains Orion technical documentation, engineering notes, handbooks,
and frequently asked questions.
```

#### Instrucciones del agente *(opcional pero recomendado)*
```
You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional,
and factual tone appropriate for engineers and technical staff. Use only verified
information from Orion's internal documents, and include source references when
available. If the answer cannot be found, clearly state that and suggest related
sections or next steps. Do not speculate, make assumptions, or provide information
outside the provided context.
```

Finalmente, haz clic en **Create Agent**.

---

### ⏳ Mientras el agente se crea (~10 minutos)...

Agent Bricks no está simplemente guardando una configuración. En este momento está:

| Paso interno | ¿Qué hace? |
|---|---|
| **Parsing** | Usa `ai_parse_document` para extraer texto, tablas e imágenes de cada PDF/DOCX |
| **Chunking** | Divide los documentos en fragmentos optimizados para retrieval |
| **Embedding** | Genera vectores para cada chunk usando un modelo de embeddings (ej. GTE) |
| **Vector Search** | Crea y popula un índice en Mosaic AI Vector Search |
| **Model Serving** | Despliega el endpoint del agente con el LLM y la lógica de retrieval |

> 🔑 **Punto clave:** Esto es lo que harías manualmente con código en el enfoque Code-First. Agent Bricks lo hace por ti, y si cambias los documentos fuente, el índice se actualiza automáticamente.

---

---
# Parte B — Probar el agente
> ⏱ **~10 minutos** · Usando la interfaz de chat de Agent Bricks

Una vez que el agente muestre estado **Ready**, usa el panel de chat integrado en la interfaz de Agent Bricks para probarlo.

### Preguntas de prueba

Prueba estas tres preguntas en orden. Cada una evalúa algo distinto:

---

**Pregunta 1** — Información técnica directa:
```
How does Orion verify compliance with ISO 13849-1?
```
*¿Qué esperar?* Una respuesta precisa con cita al documento de compliance. Observa la sección de fuentes.

---

**Pregunta 2** — Razonamiento sobre documentación técnica:
```
How does the Orion motion controller maintain stability during high-speed movement?
```
*¿Qué esperar?* El agente debe sintetizar información de los manuales de ingeniería y citar las secciones relevantes.

---

**Pregunta 3** — Pregunta fuera del contexto ⚠️:
```
What does the red blinking light on Orion mean?
```
*¿Qué esperar?* **No existe una luz roja parpadeante en la documentación de Orion.** Un buen agente debe reconocer esto y decirlo claramente, en lugar de inventar una respuesta.

---

### 🔍 Qué observar en cada respuesta

| Aspecto | ¿Qué buscar? |
|---------|-------------|
| **Citas** | ¿El agente referencia el documento y sección específica? |
| **Tono** | ¿Responde de forma profesional y técnica, como se indicó en las instrucciones? |
| **Límites** | En la pregunta 3, ¿admite que no tiene la información o intenta adivinar? |
| **Consistencia** | ¿Las respuestas 1 y 2 son coherentes con el contenido real de los documentos? |

<div style="background:#FFF3E0; border-left:4px solid #E67E22; padding:12px 16px; border-radius:4px; margin:12px 0;">
  <strong>🤔 Reflexión:</strong> ¿Qué pasó con la pregunta de la luz roja? ¿El agente respondió correctamente o intentó adivinar? Eso nos lleva a la Parte C.
</div>

---

---
# Parte C — Mejorar la calidad con ALHF
> ⏱ **~15 minutos** · Agent Learning from Human Feedback

En esta parte vamos a enseñarle al agente cómo responder mejor ante la pregunta de la luz roja. Esto es el ciclo **ALHF** en acción: feedback humano → mejora automática del sistema.

### C1 — Agregar labeled data (ejemplos con guidelines)

Este método es ideal cuando sabes exactamente qué respuesta quieres para una pregunta específica.

**Pasos:**

1. En la interfaz del agente, navega a la pestaña **Examples**
2. Haz clic en **Add**
3. Ingresa la pregunta y presiona **Add**:
   ```
   What does the red blinking light on Orion mean?
   ```
4. Haz clic en la pregunta para abrir sus detalles
5. En la sección **Guidelines**, agrega las siguientes instrucciones una por una:

   > Inform the user that there isn't a blinking red light on Orion

   > Ask the user to restart Orion by removing and reinserting the battery

   > Check the light color

   > Check the light again to see if it blinks

6. Haz clic en **Save**

**Ahora prueba el agente de nuevo** con la misma pregunta. ¿Cambió la respuesta?

También puedes variarlo preguntando:
```
What does the blue blinking light on Orion mean?
```
Observa cómo el agente adapta la respuesta al contexto nuevo usando las guidelines como guía de comportamiento.

<div style="background:#E8F5E9; border-left:4px solid #27AE60; padding:12px 16px; border-radius:4px; margin:12px 0;">
  <strong>🔑 ¿Qué acaba de pasar?</strong><br>
  Las guidelines no son respuestas hardcodeadas. Agent Bricks las usa para ajustar el comportamiento del LLM en situaciones similares. El agente aprendió un patrón, no una respuesta fija.
</div>

---

### C2 — Expert Review (revisión por expertos)

Este método es ideal para mejorar la calidad de forma continua en producción, donde los SMEs revisan respuestas reales.

El flujo de Expert Review permite:
- Revisar respuestas del agente a preguntas de evaluación
- Dar feedback en lenguaje natural sobre calidad de la respuesta
- Agregar guidelines y expectativas de comportamiento
- Evaluar respuestas en términos de exactitud, completitud y tono

Para el paso a paso completo de este proceso, consulta la documentación oficial:
[Agent Bricks: Knowledge Assistant — Step 3: Improve quality](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant#step-3-improve-quality)

<div style="background:#EDE7F6; border-left:4px solid #8E44AD; padding:12px 16px; border-radius:4px; margin:12px 0;">
  <strong>💬 Pregunta para reflexionar:</strong><br>
  ¿Cómo se reflejan las guidelines en los traces del agente? Navega a la pestaña <strong>Traces</strong> y observa cómo el sistema incorpora el feedback en el contexto que recibe el LLM.
</div>

**Mejor práctica:** La mejora de calidad es un proceso iterativo. En producción, planifica múltiples rondas de recolección de feedback y refinamiento para alcanzar el desempeño óptimo.

---

---
# 🧹 Cleanup — Eliminar recursos

> ⚠️ **Importante:** El agente KA crea un **Vector Search endpoint** y un **Model Serving endpoint** en segundo plano. Estos endpoints generan costos aunque no estén en uso activo. Elimínalos al terminar el lab.

**Pasos para eliminar el agente:**

1. Navega a **Agents** en el menú lateral
2. Localiza **Orion_Knowledge_Assistant**
3. Haz clic en el agente para abrir sus detalles
4. Selecciona **Delete** en el menú de opciones (esquina superior derecha)
5. Confirma la eliminación

Al eliminar el agente se eliminan también el Vector Search endpoint y el Model Serving endpoint asociados.

---

---
# ✅ Resumen del laboratorio

Completaste el ciclo completo de un agente Knowledge Assistant con Agent Bricks:

<div style="display:grid; grid-template-columns: 1fr 1fr 1fr; gap:16px; margin:16px 0;">

<div style="background:#E3F2FD; border-top:4px solid #1E88E5; padding:16px; border-radius:8px;">
  <div style="font-weight:700; color:#1E88E5; margin-bottom:8px;">Parte A — Crear</div>
  <div style="font-size:13px; color:#334155;">Configuraste el agente de forma declarativa: nombre, descripción, fuente de datos e instrucciones de comportamiento.</div>
</div>

<div style="background:#E8F5E9; border-top:4px solid #27AE60; padding:16px; border-radius:8px;">
  <div style="font-weight:700; color:#27AE60; margin-bottom:8px;">Parte B — Probar</div>
  <div style="font-size:13px; color:#334155;">Evaluaste el agente con preguntas dentro y fuera del contexto. Observaste citas, tono y manejo de límites del conocimiento.</div>
</div>

<div style="background:#F3E5F5; border-top:4px solid #8E44AD; padding:16px; border-radius:8px;">
  <div style="font-weight:700; color:#8E44AD; margin-bottom:8px;">Parte C — Mejorar</div>
  <div style="font-size:13px; color:#334155;">Usaste labeled data y guidelines para enseñarle al agente cómo responder mejor. Viste el ciclo ALHF en acción.</div>
</div>

</div>

## Próximos pasos

Si quieres seguir explorando Agent Bricks, considera:

1. **Expandir las fuentes de conocimiento** — Agrega más documentos al Volume y observa cómo el índice se actualiza automáticamente
2. **Explorar el Multi-Agent Supervisor** — Combina OKA con un Genie Agent para responder tanto preguntas sobre documentos como consultas sobre datos estructurados
3. **Revisar los traces en MLflow** — Cada interacción queda trazada; puedes analizar latencia, retrieval scores y calidad de respuestas

## Recursos

| Recurso | Link |
|---------|------|
| Documentación Agent Bricks | [docs.databricks.com](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant) |
| Mosaic AI Agent Evaluation | [docs.databricks.com](https://docs.databricks.com/aws/en/generative-ai/agent-evaluation/index.html) |
| MLflow Tracing | [mlflow.org](https://mlflow.org/docs/latest/llms/tracing/index.html) |

---

<div style="text-align:center; color:#8BA3BE; font-size:12px; margin-top:24px;">
  © 2026 · Laboratorio preparado para clase demo · Knowledge Assistant con Agent Bricks
</div>